# Validación P2 — HPO y paralelismo CPU/GPU
Elegir GPU en Colab si está disponible. Subir ZIP y copiar la carpeta HPO anterior o activar ambas etapas. CSV 7–10 reservados.


In [ ]:
from google.colab import files
import zipfile, pathlib
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
with zipfile.ZipFile(archive) as z:
    z.extractall("/content/automl_p2")
PROJECT_DIR = str(next(pathlib.Path("/content/automl_p2").rglob("main.py")).parent)
%cd $PROJECT_DIR
%pip install -q -r requirements.txt


In [ ]:
import yaml
from pathlib import Path
config = yaml.safe_load(Path("config/config.yaml").read_text())
# True/True: flujo completo. False/True: reutilizar HPO con ruta existente.
config["modules"] = {"hpo": True, "parallel": True, "backtesting": True}
config["parallel"]["workers"] = 2
# config["parallel"]["hpo_run_dir"] = "/content/drive/MyDrive/CARPETA_CORRIDA_HPO"
Path("config/config.yaml").write_text(yaml.safe_dump(config, sort_keys=False))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
INPUT_DIR = "/content/drive/MyDrive/REEMPLAZAR_CARPETA_CSV"
import subprocess, sys
subprocess.run([sys.executable, "main.py", "--input", INPUT_DIR, "--config", "config/config.yaml"], check=True)

In [ ]:
import json
from pathlib import Path
runs = sorted(Path("data/output/parallel").glob("*/results_summary.json"))
assert runs, "No se generó benchmark"
summary = json.loads(runs[-1].read_text())
assert not summary["data_split"]["future_files_loaded"]
assert len(summary["models"]) == 9
assert summary["speedup"] > 0
if summary["hardware"]["available"]:
    assert any(v["device"] == "GPU" for v in summary["device_plan"].values())
print("Secuencial:", summary["T_sequential_seconds"])
print("Paralelo:", summary["T_parallel_seconds"])
print("Speed-up:", summary["speedup"])
print("Dispositivos:", summary["device_plan"])


In [ ]:
from pathlib import Path
import json
runs = sorted(Path("data/output/backtesting").glob("*/backtesting_results.json"))
assert runs
back = json.loads(runs[-1].read_text(encoding="utf-8"))
assert len(back["waves"]) >= 4
assert not back["model_refitted"]
print("Backtesting:", back["status"])
from IPython.display import display, Image
display(Image(filename=str(runs[-1].parent / "auc_por_ola.png")))
